In [10]:
pip install yfinance

In [11]:
import yfinance as yf
import numpy as np
import pandas as pd
from datetime import datetime
import statsmodels.api as sm



def validar_y_convertir_fechas(fechas):
    """
    Convierte una lista de fechas a formato 'YYYY-MM-DD' si no están ya en ese formato.

    Parámetros:
    - fechas: Lista de fechas como cadenas.

    Retorna:
    - Lista de fechas en formato 'YYYY-MM-DD'.
    """
    fechas_convertidas = []
    for fecha in fechas:
        try:
            # Intenta parsear la fecha asumiendo el formato 'YYYY-MM-DD'
            fecha_convertida = datetime.strptime(fecha, '%Y-%m-%d')
        except ValueError:
            # Intenta otros formatos si el anterior falla
            fecha_convertida = pd.to_datetime(fecha, errors='coerce')
            if pd.isnull(fecha_convertida):
                raise ValueError(f"Fecha no reconocida: {fecha}")
        # Asegura el formato correcto
        fechas_convertidas.append(fecha_convertida.strftime('%Y-%m-%d'))
    return fechas_convertidas

def obtener_precios_logaritmicos(ticker, fechas):
    """
    Descarga los datos de precios de un ticker específico y calcula la variación logarítmica
    de los precios de cierre para las fechas dadas, y luego formatea los resultados en porcentaje.

    Parámetros:
    - ticker: El símbolo del ticker de la acción (como string).
    - fechas: Lista de fechas en formato 'YYYY-MM-DD'.

    Retorna:
    - Un DataFrame con las fechas dadas y las variaciones logarítmicas de los precios de cierre entre ellas,
      formateadas en porcentaje con el símbolo '%'.
    """
    # Descargar datos de un rango que cubra las fechas dadas
    datos = yf.download(ticker, start=min(fechas), end=max(fechas))

    # Asegurarse que las fechas son tratadas como datetime
    fechas = pd.to_datetime(fechas)

    # Reindexar los datos para incluir todas las fechas dadas, llenando hacia adelante para obtener el precio más reciente si una fecha no es un día de trading
    datos_reindexados = datos.reindex(fechas, method='bfill')

    # Seleccionar solo la columna 'Close'
    precios_cierre = datos_reindexados['Close']

    # Calcular la variación logarítmica
    variacion_log = np.log(precios_cierre).diff().dropna()

    # Convertir a porcentaje, ajustar formato decimal y añadir símbolo de porcentaje
    variacion_log_porcentaje = variacion_log.apply(lambda x: f"{x*100:.2f}".replace('.', ',') + '%')

    # Convertir el índice a DatetimeIndex y renombrarlo
    variacion_log_porcentaje.index = pd.to_datetime(variacion_log_porcentaje.index)
    variacion_log_porcentaje.index.name = 'Date'

    # Ahora crea el DataFrame
    resultado = pd.DataFrame({
        'Variacion Logaritmica': variacion_log_porcentaje.values
    }, index=variacion_log_porcentaje.index)

    return resultado




In [20]:

def regresion(ticker):


  fechas = ["2000-01-07", "2000-01-14", "2000-01-21", "2000-01-28", "2000-02-04", "2000-02-11", "2000-02-18", "2000-02-25", "2000-03-03", "2000-03-10", "2000-03-17", "2000-03-24", "2000-03-31", "2000-04-07", "2000-04-14", "2000-04-20", "2000-04-28", "2000-05-05", "2000-05-12", "2000-05-19", "2000-05-26", "2000-06-02", "2000-06-09", "2000-06-16", "2000-06-23", "2000-06-30", "2000-07-07", "2000-07-14", "2000-07-21", "2000-07-28", "2000-08-04", "2000-08-11", "2000-08-18", "2000-08-25", "2000-09-01", "2000-09-08", "2000-09-15", "2000-09-22", "2000-09-29", "2000-10-06", "2000-10-13", "2000-10-20", "2000-10-27", "2000-11-03", "2000-11-10", "2000-11-17", "2000-11-24", "2000-12-01", "2000-12-08", "2000-12-15", "2000-12-22", "2000-12-29", "2001-01-05", "2001-01-12", "2001-01-19", "2001-01-26", "2001-02-02", "2001-02-09", "2001-02-16", "2001-02-23", "2001-03-02", "2001-03-09", "2001-03-16", "2001-03-23", "2001-03-30", "2001-04-06", "2001-04-12", "2001-04-20", "2001-04-27", "2001-05-04", "2001-05-11", "2001-05-18", "2001-05-25", "2001-06-01", "2001-06-08", "2001-06-15", "2001-06-22", "2001-06-29", "2001-07-06", "2001-07-13", "2001-07-20", "2001-07-27", "2001-08-03", "2001-08-10", "2001-08-17", "2001-08-24", "2001-08-31", "2001-09-07", "2001-09-10", "2001-09-21", "2001-09-28", "2001-10-05", "2001-10-12", "2001-10-19", "2001-10-26", "2001-11-02", "2001-11-09", "2001-11-16", "2001-11-23", "2001-11-30", "2001-12-07", "2001-12-14", "2001-12-21", "2001-12-28", "2002-01-04", "2002-01-11", "2002-01-18", "2002-01-25", "2002-02-01", "2002-02-08", "2002-02-15", "2002-02-22", "2002-03-01", "2002-03-08", "2002-03-15", "2002-03-22", "2002-03-28", "2002-04-05", "2002-04-12", "2002-04-19", "2002-04-26", "2002-05-03", "2002-05-10", "2002-05-17", "2002-05-24", "2002-05-31", "2002-06-07", "2002-06-14", "2002-06-21", "2002-06-28", "2002-07-05", "2002-07-12", "2002-07-19", "2002-07-26", "2002-08-02", "2002-08-09", "2002-08-16", "2002-08-23", "2002-08-30", "2002-09-06", "2002-09-13", "2002-09-20", "2002-09-27", "2002-10-04", "2002-10-11", "2002-10-18", "2002-10-25", "2002-11-01", "2002-11-08", "2002-11-15", "2002-11-22", "2002-11-29", "2002-12-06", "2002-12-13", "2002-12-20", "2002-12-27", "2003-01-03", "2003-01-10", "2003-01-17", "2003-01-24", "2003-01-31", "2003-02-07", "2003-02-14", "2003-02-21", "2003-02-28", "2003-03-07", "2003-03-14", "2003-03-21", "2003-03-28", "2003-04-04", "2003-04-11", "2003-04-17", "2003-04-25", "2003-05-02", "2003-05-09", "2003-05-16", "2003-05-23", "2003-05-30", "2003-06-06", "2003-06-13", "2003-06-20", "2003-06-27", "2003-07-03", "2003-07-11", "2003-07-18", "2003-07-25", "2003-08-01", "2003-08-08", "2003-08-15", "2003-08-22", "2003-08-29", "2003-09-05", "2003-09-12", "2003-09-19", "2003-09-26", "2003-10-03", "2003-10-10", "2003-10-17", "2003-10-24", "2003-10-31", "2003-11-07", "2003-11-14", "2003-11-21", "2003-11-28", "2003-12-05", "2003-12-12", "2003-12-19", "2003-12-26", "2004-01-02", "2004-01-09", "2004-01-16", "2004-01-23", "2004-01-30", "2004-02-06", "2004-02-13", "2004-02-20", "2004-02-27", "2004-03-05", "2004-03-12", "2004-03-19", "2004-03-26", "2004-04-02", "2004-04-08", "2004-04-16", "2004-04-23", "2004-04-30", "2004-05-07", "2004-05-14", "2004-05-21", "2004-05-28", "2004-06-04", "2004-06-10", "2004-06-18", "2004-06-25", "2004-07-02", "2004-07-09", "2004-07-16", "2004-07-23", "2004-07-30", "2004-08-06", "2004-08-13", "2004-08-20", "2004-08-27", "2004-09-03", "2004-09-10", "2004-09-17", "2004-09-24", "2004-10-01", "2004-10-08", "2004-10-15", "2004-10-22", "2004-10-29", "2004-11-05", "2004-11-12", "2004-11-19", "2004-11-26", "2004-12-03", "2004-12-10", "2004-12-17", "2004-12-23", "2004-12-31", "2005-01-07", "2005-01-14", "2005-01-21", "2005-01-28", "2005-02-04", "2005-02-11", "2005-02-18", "2005-02-25", "2005-03-04", "2005-03-11", "2005-03-18", "2005-03-24", "2005-04-01", "2005-04-08", "2005-04-15", "2005-04-22", "2005-04-29", "2005-05-06", "2005-05-13", "2005-05-20", "2005-05-27", "2005-06-03", "2005-06-10", "2005-06-17", "2005-06-24", "2005-07-01", "2005-07-08", "2005-07-15", "2005-07-22", "2005-07-29", "2005-08-05", "2005-08-12", "2005-08-19", "2005-08-26", "2005-09-02", "2005-09-09", "2005-09-16", "2005-09-23", "2005-09-30", "2005-10-07", "2005-10-14", "2005-10-21", "2005-10-28", "2005-11-04", "2005-11-11", "2005-11-18", "2005-11-25", "2005-12-02", "2005-12-09", "2005-12-16", "2005-12-23", "2005-12-30", "2006-01-06", "2006-01-13", "2006-01-20", "2006-01-27", "2006-02-03", "2006-02-10", "2006-02-17", "2006-02-24", "2006-03-03", "2006-03-10", "2006-03-17", "2006-03-24", "2006-03-31", "2006-04-07", "2006-04-13", "2006-04-21", "2006-04-28", "2006-05-05", "2006-05-12", "2006-05-19", "2006-05-26", "2006-06-02", "2006-06-09", "2006-06-16", "2006-06-23", "2006-06-30", "2006-07-07", "2006-07-14", "2006-07-21", "2006-07-28", "2006-08-04", "2006-08-11", "2006-08-18", "2006-08-25", "2006-09-01", "2006-09-08", "2006-09-15", "2006-09-22", "2006-09-29", "2006-10-06", "2006-10-13", "2006-10-20", "2006-10-27", "2006-11-03", "2006-11-10", "2006-11-17", "2006-11-24", "2006-12-01", "2006-12-08", "2006-12-15", "2006-12-22", "2006-12-29", "2007-01-05", "2007-01-12", "2007-01-19", "2007-01-26", "2007-02-02", "2007-02-09", "2007-02-16", "2007-02-23", "2007-03-02", "2007-03-09", "2007-03-16", "2007-03-23", "2007-03-30", "2007-04-05", "2007-04-13", "2007-04-20", "2007-04-27", "2007-05-04", "2007-05-11", "2007-05-18", "2007-05-25", "2007-06-01", "2007-06-08", "2007-06-15", "2007-06-22", "2007-06-29", "2007-07-06", "2007-07-13", "2007-07-20", "2007-07-27", "2007-08-03", "2007-08-10", "2007-08-17", "2007-08-24", "2007-08-31", "2007-09-07", "2007-09-14", "2007-09-21", "2007-09-28", "2007-10-05", "2007-10-12", "2007-10-19", "2007-10-26", "2007-11-02", "2007-11-09", "2007-11-16", "2007-11-23", "2007-11-30", "2007-12-07", "2007-12-14", "2007-12-21", "2007-12-28", "2008-01-04", "2008-01-11", "2008-01-18", "2008-01-25", "2008-02-01", "2008-02-08", "2008-02-15", "2008-02-22", "2008-02-29", "2008-03-07", "2008-03-14", "2008-03-20", "2008-03-28", "2008-04-04", "2008-04-11", "2008-04-18", "2008-04-25", "2008-05-02", "2008-05-09", "2008-05-16", "2008-05-23", "2008-05-30", "2008-06-06", "2008-06-13", "2008-06-20", "2008-06-27", "2008-07-03", "2008-07-11", "2008-07-18", "2008-07-25", "2008-08-01", "2008-08-08", "2008-08-15", "2008-08-22", "2008-08-29", "2008-09-05", "2008-09-12", "2008-09-19", "2008-09-26", "2008-10-03", "2008-10-10", "2008-10-17", "2008-10-24", "2008-10-31", "2008-11-07", "2008-11-14", "2008-11-21", "2008-11-28", "2008-12-05", "2008-12-12", "2008-12-19", "2008-12-26", "2009-01-02", "2009-01-09", "2009-01-16", "2009-01-23", "2009-01-30", "2009-02-06", "2009-02-13", "2009-02-20", "2009-02-27", "2009-03-06", "2009-03-13", "2009-03-20", "2009-03-27", "2009-04-03", "2009-04-09", "2009-04-17", "2009-04-24", "2009-05-01", "2009-05-08", "2009-05-15", "2009-05-22", "2009-05-29", "2009-06-05", "2009-06-12", "2009-06-19", "2009-06-26", "2009-07-02", "2009-07-10", "2009-07-17", "2009-07-24", "2009-07-31", "2009-08-07", "2009-08-14", "2009-08-21", "2009-08-28", "2009-09-04", "2009-09-11", "2009-09-18", "2009-09-25", "2009-10-02", "2009-10-09", "2009-10-16", "2009-10-23", "2009-10-30", "2009-11-06", "2009-11-13", "2009-11-20", "2009-11-27", "2009-12-04", "2009-12-11", "2009-12-18", "2009-12-24", "2009-12-31", "2010-01-08", "2010-01-15", "2010-01-22", "2010-01-29", "2010-02-05", "2010-02-12", "2010-02-19", "2010-02-26", "2010-03-05", "2010-03-12", "2010-03-19", "2010-03-26", "2010-04-01", "2010-04-09", "2010-04-16", "2010-04-23", "2010-04-30", "2010-05-07", "2010-05-14", "2010-05-21", "2010-05-28", "2010-06-04", "2010-06-11", "2010-06-18", "2010-06-25", "2010-07-02", "2010-07-09", "2010-07-16", "2010-07-23", "2010-07-30", "2010-08-06", "2010-08-13", "2010-08-20", "2010-08-27", "2010-09-03", "2010-09-10", "2010-09-17", "2010-09-24", "2010-10-01", "2010-10-08", "2010-10-15", "2010-10-22", "2010-10-29", "2010-11-05", "2010-11-12", "2010-11-19", "2010-11-26", "2010-12-03", "2010-12-10", "2010-12-17", "2010-12-23", "2010-12-31", "2011-01-07", "2011-01-14", "2011-01-21", "2011-01-28", "2011-02-04", "2011-02-11", "2011-02-18", "2011-02-25", "2011-03-04", "2011-03-11", "2011-03-18", "2011-03-25", "2011-04-01", "2011-04-08", "2011-04-15", "2011-04-21", "2011-04-29", "2011-05-06", "2011-05-13", "2011-05-20", "2011-05-27", "2011-06-03", "2011-06-10", "2011-06-17", "2011-06-24", "2011-07-01", "2011-07-08", "2011-07-15", "2011-07-22", "2011-07-29", "2011-08-05", "2011-08-12", "2011-08-19", "2011-08-26", "2011-09-02", "2011-09-09", "2011-09-16", "2011-09-23", "2011-09-30", "2011-10-07", "2011-10-14", "2011-10-21", "2011-10-28", "2011-11-04", "2011-11-11", "2011-11-18", "2011-11-25", "2011-12-02", "2011-12-09", "2011-12-16", "2011-12-23", "2011-12-30", "2012-01-06", "2012-01-13", "2012-01-20", "2012-01-27", "2012-02-03", "2012-02-10", "2012-02-17", "2012-02-24", "2012-03-02", "2012-03-09", "2012-03-16", "2012-03-23", "2012-03-30", "2012-04-05", "2012-04-13", "2012-04-20", "2012-04-27", "2012-05-04", "2012-05-11", "2012-05-18", "2012-05-25", "2012-06-01", "2012-06-08", "2012-06-15", "2012-06-22", "2012-06-29", "2012-07-06", "2012-07-13", "2012-07-20", "2012-07-27", "2012-08-03", "2012-08-10", "2012-08-17", "2012-08-24", "2012-08-31", "2012-09-07", "2012-09-14", "2012-09-21", "2012-09-28", "2012-10-05", "2012-10-12", "2012-10-19", "2012-10-26", "2012-11-02", "2012-11-09", "2012-11-16", "2012-11-23", "2012-11-30", "2012-12-07", "2012-12-14", "2012-12-21", "2012-12-28", "2013-01-04", "2013-01-11", "2013-01-18", "2013-01-25", "2013-02-01", "2013-02-08", "2013-02-15", "2013-02-22", "2013-03-01", "2013-03-08", "2013-03-15", "2013-03-22", "2013-03-28", "2013-04-05", "2013-04-12", "2013-04-19", "2013-04-26", "2013-05-03", "2013-05-10", "2013-05-17", "2013-05-24", "2013-05-31", "2013-06-07", "2013-06-14", "2013-06-21", "2013-06-28", "2013-07-05", "2013-07-12", "2013-07-19", "2013-07-26", "2013-08-02", "2013-08-09", "2013-08-16", "2013-08-23", "2013-08-30", "2013-09-06", "2013-09-13", "2013-09-20", "2013-09-27", "2013-10-04", "2013-10-11", "2013-10-18", "2013-10-25", "2013-11-01", "2013-11-08", "2013-11-15", "2013-11-22", "2013-11-29", "2013-12-06", "2013-12-13", "2013-12-20", "2013-12-27", "2014-01-03", "2014-01-10", "2014-01-17", "2014-01-24", "2014-01-31", "2014-02-07", "2014-02-14", "2014-02-21", "2014-02-28", "2014-03-07", "2014-03-14", "2014-03-21", "2014-03-28", "2014-04-04", "2014-04-11", "2014-04-17", "2014-04-25", "2014-05-02", "2014-05-09", "2014-05-16", "2014-05-23", "2014-05-30", "2014-06-06", "2014-06-13", "2014-06-20", "2014-06-27", "2014-07-03", "2014-07-11", "2014-07-18", "2014-07-25", "2014-08-01", "2014-08-08", "2014-08-15", "2014-08-22", "2014-08-29", "2014-09-05", "2014-09-12", "2014-09-19", "2014-09-26", "2014-10-03", "2014-10-10", "2014-10-17", "2014-10-24", "2014-10-31", "2014-11-07", "2014-11-14", "2014-11-21", "2014-11-28", "2014-12-05", "2014-12-12", "2014-12-19", "2014-12-26", "2015-01-02", "2015-01-09", "2015-01-16", "2015-01-23", "2015-01-30", "2015-02-06", "2015-02-13", "2015-02-20", "2015-02-27", "2015-03-06", "2015-03-13", "2015-03-20", "2015-03-27", "2015-04-02", "2015-04-10", "2015-04-17", "2015-04-24", "2015-05-01", "2015-05-08", "2015-05-15", "2015-05-22", "2015-05-29", "2015-06-05", "2015-06-12", "2015-06-19", "2015-06-26", "2015-07-02", "2015-07-10", "2015-07-17", "2015-07-24", "2015-07-31", "2015-08-07", "2015-08-14", "2015-08-21", "2015-08-28", "2015-09-04", "2015-09-11", "2015-09-18", "2015-09-25", "2015-10-02", "2015-10-09", "2015-10-16", "2015-10-23", "2015-10-30", "2015-11-06", "2015-11-13", "2015-11-20", "2015-11-27", "2015-12-04", "2015-12-11", "2015-12-18", "2015-12-24", "2015-12-31", "2016-01-08", "2016-01-15", "2016-01-22", "2016-01-29", "2016-02-05", "2016-02-12", "2016-02-19", "2016-02-26", "2016-03-04", "2016-03-11", "2016-03-18", "2016-03-24", "2016-04-01", "2016-04-08", "2016-04-15", "2016-04-22", "2016-04-29", "2016-05-06", "2016-05-13", "2016-05-20", "2016-05-27", "2016-06-03", "2016-06-10", "2016-06-17", "2016-06-24", "2016-07-01", "2016-07-08", "2016-07-15", "2016-07-22", "2016-07-29", "2016-08-05", "2016-08-12", "2016-08-19", "2016-08-26", "2016-09-02", "2016-09-09", "2016-09-16", "2016-09-23", "2016-09-30", "2016-10-07", "2016-10-14", "2016-10-21", "2016-10-28", "2016-11-04", "2016-11-11", "2016-11-18", "2016-11-25", "2016-12-02", "2016-12-09", "2016-12-16", "2016-12-23", "2016-12-30", "2017-01-06", "2017-01-13", "2017-01-20", "2017-01-27", "2017-02-03", "2017-02-10", "2017-02-17", "2017-02-24", "2017-03-03", "2017-03-10", "2017-03-17", "2017-03-24", "2017-03-31", "2017-04-07", "2017-04-13", "2017-04-21", "2017-04-28", "2017-05-05", "2017-05-12", "2017-05-19", "2017-05-26", "2017-06-02", "2017-06-09", "2017-06-16", "2017-06-23", "2017-06-30", "2017-07-07", "2017-07-14", "2017-07-21", "2017-07-28", "2017-08-04", "2017-08-11", "2017-08-18", "2017-08-25", "2017-09-01", "2017-09-08", "2017-09-15", "2017-09-22", "2017-09-29", "2017-10-06", "2017-10-13", "2017-10-20", "2017-10-27", "2017-11-03", "2017-11-10", "2017-11-17", "2017-11-24", "2017-12-01", "2017-12-08", "2017-12-15", "2017-12-22", "2017-12-29", "2018-01-05", "2018-01-12", "2018-01-19", "2018-01-26", "2018-02-02", "2018-02-09", "2018-02-16", "2018-02-23", "2018-03-02", "2018-03-09", "2018-03-16", "2018-03-23", "2018-03-29", "2018-04-06", "2018-04-13", "2018-04-20", "2018-04-27", "2018-05-04", "2018-05-11", "2018-05-18", "2018-05-25", "2018-06-01", "2018-06-08", "2018-06-15", "2018-06-22", "2018-06-29", "2018-07-06", "2018-07-13", "2018-07-20", "2018-07-27", "2018-08-03", "2018-08-10", "2018-08-17", "2018-08-24", "2018-08-31", "2018-09-07", "2018-09-14", "2018-09-21", "2018-09-28", "2018-10-05", "2018-10-12", "2018-10-19", "2018-10-26", "2018-11-02", "2018-11-09", "2018-11-16", "2018-11-23", "2018-11-30", "2018-12-07", "2018-12-14", "2018-12-21", "2018-12-28", "2019-01-04", "2019-01-11", "2019-01-18", "2019-01-25", "2019-02-01", "2019-02-08", "2019-02-15", "2019-02-22", "2019-03-01", "2019-03-08", "2019-03-15", "2019-03-22", "2019-03-29", "2019-04-05", "2019-04-12", "2019-04-18", "2019-04-26", "2019-05-03", "2019-05-10", "2019-05-17", "2019-05-24", "2019-05-31", "2019-06-07", "2019-06-14", "2019-06-21", "2019-06-28", "2019-07-05", "2019-07-12", "2019-07-19", "2019-07-26", "2019-08-02", "2019-08-09", "2019-08-16", "2019-08-23", "2019-08-30", "2019-09-06", "2019-09-13", "2019-09-20", "2019-09-27", "2019-10-04", "2019-10-11", "2019-10-18", "2019-10-25", "2019-11-01", "2019-11-08", "2019-11-15", "2019-11-22", "2019-11-29", "2019-12-06", "2019-12-13", "2019-12-20", "2019-12-27", "2020-01-03", "2020-01-10", "2020-01-17", "2020-01-24", "2020-01-31", "2020-02-07", "2020-02-14", "2020-02-21", "2020-02-28", "2020-03-06", "2020-03-13", "2020-03-20", "2020-03-27", "2020-04-03", "2020-04-09", "2020-04-17", "2020-04-24", "2020-05-01", "2020-05-08", "2020-05-15", "2020-05-22", "2020-05-29", "2020-06-05", "2020-06-12", "2020-06-19", "2020-06-26", "2020-07-02", "2020-07-10", "2020-07-17", "2020-07-24", "2020-07-31", "2020-08-07", "2020-08-14", "2020-08-21", "2020-08-28", "2020-09-04", "2020-09-11", "2020-09-18", "2020-09-25", "2020-10-02", "2020-10-09", "2020-10-16", "2020-10-23", "2020-10-30", "2020-11-06", "2020-11-13", "2020-11-20", "2020-11-27", "2020-12-04", "2020-12-11", "2020-12-18", "2020-12-24", "2020-12-31", "2021-01-08", "2021-01-15", "2021-01-22", "2021-01-29", "2021-02-05", "2021-02-12", "2021-02-19", "2021-02-26", "2021-03-05", "2021-03-12", "2021-03-19", "2021-03-26", "2021-04-01", "2021-04-09", "2021-04-16", "2021-04-23", "2021-04-30", "2021-05-07", "2021-05-14", "2021-05-21", "2021-05-28", "2021-06-04", "2021-06-11", "2021-06-18", "2021-06-25", "2021-07-02", "2021-07-09", "2021-07-16", "2021-07-23", "2021-07-30", "2021-08-06", "2021-08-13", "2021-08-20", "2021-08-27", "2021-09-03", "2021-09-10", "2021-09-17", "2021-09-24", "2021-10-01", "2021-10-08", "2021-10-15", "2021-10-22", "2021-10-29", "2021-11-05", "2021-11-12", "2021-11-19", "2021-11-26", "2021-12-03", "2021-12-10", "2021-12-17", "2021-12-23", "2021-12-31", "2022-01-07", "2022-01-14", "2022-01-21", "2022-01-28", "2022-02-04", "2022-02-11", "2022-02-18", "2022-02-25", "2022-03-04", "2022-03-11", "2022-03-18", "2022-03-25", "2022-04-01", "2022-04-08", "2022-04-14", "2022-04-22", "2022-04-29", "2022-05-06", "2022-05-13", "2022-05-20", "2022-05-27", "2022-06-03", "2022-06-10", "2022-06-17", "2022-06-24", "2022-07-01", "2022-07-08", "2022-07-15", "2022-07-22", "2022-07-29", "2022-08-05", "2022-08-12", "2022-08-19", "2022-08-26", "2022-09-02", "2022-09-09", "2022-09-16", "2022-09-23", "2022-09-30", "2022-10-07", "2022-10-14", "2022-10-21", "2022-10-28", "2022-11-04", "2022-11-11", "2022-11-18", "2022-11-25", "2022-12-02", "2022-12-09", "2022-12-16", "2022-12-23", "2022-12-30", "2023-01-06", "2023-01-13", "2023-01-20", "2023-01-27", "2023-02-03", "2023-02-10", "2023-02-17", "2023-02-24", "2023-03-03", "2023-03-10", "2023-03-17", "2023-03-24", "2023-03-31", "2023-04-06", "2023-04-14", "2023-04-21", "2023-04-28", "2023-05-05", "2023-05-12", "2023-05-19", "2023-05-26", "2023-06-02", "2023-06-09", "2023-06-16", "2023-06-23", "2023-06-30", "2023-07-07", "2023-07-14", "2023-07-21", "2023-07-28", "2023-08-04", "2023-08-11", "2023-08-18", "2023-08-25", "2023-09-01", "2023-09-08", "2023-09-15", "2023-09-22", "2023-09-29", "2023-10-06", "2023-10-13", "2023-10-20", "2023-10-27", "2023-11-03", "2023-11-10", "2023-11-17", "2023-11-24", "2023-12-01", "2023-12-08", "2023-12-15", "2023-12-22", "2023-12-29", "2024-01-05", "2024-01-12", "2024-01-19", "2024-01-26", "2024-02-02", "2024-02-09", "2024-02-16", "2024-02-23",]


  # Ejemplo de uso
  # Símbolo del ticker para Apple Inc.

  fechas = validar_y_convertir_fechas(fechas)


  df_variacion_log = obtener_precios_logaritmicos(ticker,fechas)
  df_variacion_log.head()




  famafrench_df = pd.read_csv('/content/general_csv_weekly.csv', sep = ';')


  # Convierte la columna de fecha a datetime
  famafrench_df['Date'] = pd.to_datetime(famafrench_df['Date'], format="%Y-%m-%d")

  # Si necesitas un formato específico, puedes usar el parámetro 'format'
  # df['Fecha'] = pd.to_datetime(df['Fecha'], format='%Y-%m-%d')

  # Después de convertir, establece la columna de fecha como índice si deseas
  famafrench_df.set_index('Date', inplace=True)



  famafrench_df.head()


  # Si ambos DataFrames tienen el índice de fecha correctamente configurado, puedes proceder directamente a merge
  df_combinado = famafrench_df.merge(df_variacion_log, left_index=True, right_index=True, how='outer')
  # Eliminar filas que contengan algún valor NaN
  df_combinado = df_combinado.dropna()

  # Convertir el índice de fecha a una columna regular

  df_combinado = df_combinado.round(3)

  # Asegúrate de que las columnas son tratadas como strings antes de reemplazar ',' por '.'
  df_combinado['Mkt-RF'] = df_combinado['Mkt-RF'].astype(str).str.replace(',', '.').astype(float)
  df_combinado['SMB'] = df_combinado['SMB'].astype(str).str.replace(',', '.').astype(float)
  df_combinado['HML'] = df_combinado['HML'].astype(str).str.replace(',', '.').astype(float)


  # Asegura que Pandas trate las columnas como strings antes de realizar operaciones de strings
  df_combinado['RF'] = df_combinado['RF'].astype(str).str.rstrip('%').str.replace(',', '.').astype(float) / 100
  df_combinado['Variacion Logaritmica'] = df_combinado['Variacion Logaritmica'].astype(str).str.rstrip('%').str.replace(',', '.').astype(float) / 100

  # Preparar las variables independientes
  # Añadir una constante al modelo para el término de intercepción
  X = df_combinado[['Mkt-RF', 'SMB', 'HML']]
  X = sm.add_constant(X)

  # La variable dependiente es el exceso de rendimiento de Apple
  # Asegúrate de que 'RF' y 'rendimientos apple' están en formato decimal adecuado
  Y = df_combinado['Variacion Logaritmica'] - df_combinado['RF']

  # Estimar el modelo OLS
  modelo = sm.OLS(Y, X).fit()


  # Extraer el p-valor de la constante del modelo
  p_valor_const = modelo.pvalues['const']

  return p_valor_const






In [23]:
# Asume que esta es tu lista de tickers
lista_tickers = ["MSFT", "AAPL", "NVDA", "AMZN", "META", "GOOGL", "GOOG", "BRK.B", "LLY", "AVGO", "JPM", "TSLA", "XOM", "V", "UNH", "MA", "PG", "JNJ", "HD", "MRK", "COST", "ABBV", "CRM", "CVX", "AMD", "NFLX", "BAC", "WMT", "PEP", "KO", "LIN", "TMO", "ADBE", "DIS", "ACN", "WFC", "ORCL", "CSCO", "MCD", "QCOM", "ABT", "CAT", "INTU", "AMAT", "IBM", "VZ", "GE", "CMCSA", "NOW", "INTC", "DHR", "COP", "UBER", "TXN", "PFE", "UNP", "AMGN", "PM", "LOW", "SPGI", "ISRG", "MU", "RTX", "GS", "NEE", "HON", "ETN", "AXP", "LRCX", "BKNG", "PGR", "T", "ELV", "SYK", "C", "MS", "PLD", "BLK", "MDT", "TJX", "NKE", "UPS", "SCHW", "DE", "CI", "BA", "VRTX", "BMY", "CB", "ADP", "MMC", "BSX", "REGN", "SBUX", "ADI", "LMT", "FI", "KLAC", "CVS", "BX", "MDLZ", "AMT", "SNPS", "GILD", "PANW", "CDNS", "TMUS", "CMG", "MPC", "EOG", "ICE", "TGT", "SHW", "SLB", "CME", "SO", "ZTS", "WM", "ANET", "DUK", "MO", "EQIX", "PH", "PSX", "CL", "ITW", "FCX", "PYPL", "CSX", "BDX", "MCK", "ABNB", "APH", "TT", "TDG", "USB", "GD", "ORLY", "EMR", "HCA", "NOC", "PNC", "PCAR", "AON", "FDX", "PXD", "NXPI", "MAR", "MCO", "VLO", "CEG", "CTAS", "MSI", "ROP", "ECL", "NSC", "EW", "COF", "AIG", "DXCM", "HLT", "AZO", "APD", "F", "TRV", "AJG", "ADSK", "TFC", "GM", "WELL", "MMM", "NUE", "SPG", "CPRT", "CARR", "MCHP", "URI", "ROST", "WMB", "DHI", "SMCI", "OKE", "PSA", "NEM", "OXY", "MET", "AFL", "ALL", "TEL", "GWW", "SRE", "O", "AEP", "IQV", "JCI", "AMP", "FTNT", "CCI", "MSCI", "DLR", "FAST", "FIS", "BK", "HES", "STZ", "IDXX", "KMB", "A", "DOW", "AME", "PRU", "LULU", "LEN", "MNST", "CMI", "D", "CTVA", "ODFL", "OTIS", "COR", "PAYX", "LHX", "GIS", "HUM", "CNC", "SYY", "RSG", "MLM", "CSGP", "PWR", "IR", "YUM", "EXC", "GEHC", "FANG", "IT", "HAL", "KR", "PCG", "VMC", "CTSH", "KMI", "GEV", "ACGL", "MRNA", "KVUE", "DG", "BKR", "DVN", "CDW", "EL", "ADM", "GPN", "PEG", "PPG", "VRSK", "DD", "RCL", "MPWR", "ROK", "KDP", "EA", "EFX", "EXR", "DFS", "ED", "HIG", "VICI", "FICO", "XYL", "DAL", "ANSS", "XEL", "BIIB", "FTV", "ON", "KHC", "HSY", "WST", "CBRE", "MTD", "KEYS", "WTW", "RMD", "EIX", "CHTR", "TSCO", "CAH", "WAB", "EBAY", "DLTR", "ZBH", "LYB", "TROW", "AVB", "HWM", "TRGP", "WEC", "HPQ", "WY", "NVR", "CHD", "PHM", "BLDR", "FITB", "DOV", "GLW", "RJF", "TTWO", "BR", "NDAQ", "STT", "WDC", "MTB", "HPE", "AWK", "IRM", "SBAC", "GRMN", "ALGN", "DECK", "DTE", "STLD", "ETR", "HUBB", "ULTA", "PTC", "MOH", "CPAY", "NTAP", "AXON", "EQR", "IFF", "APTV", "BAX", "GPC", "CTRA", "STE", "BALL", "ES", "ILMN", "INVH", "BRO", "PPL", "HBAN", "WAT", "FE", "ARE", "COO", "TDY", "LVS", "CBOE", "VLTO", "FSLR", "CINF", "AEE", "TXT", "MKC", "RF", "WBD", "DRI", "PFG", "J", "OMC", "NTRS", "HOLX", "IEX", "CLX", "CNP", "LH", "JBL", "WRB", "LDOS", "AVY", "EXPE", "SYF", "DPZ", "TYL", "VTR", "MAS", "ATO", "CMS", "MRO", "STX", "EXPD", "PKG", "LUV", "TSN", "FDS", "NRG", "SWKS", "VRSN", "TER", "EG", "CE", "CFG", "AKAM", "JBHT", "CCL", "ENPH", "ESS", "BBY", "SNA", "TRMB", "ALB", "BG", "EPAM", "MAA", "POOL", "CF", "ZBRA", "K", "EQT", "CAG", "SWK", "NDSN", "LYV", "DGX", "HST", "KEY", "UAL", "VTRS", "L", "LKQ", "WBA", "PNR", "DOC", "IP", "AMCR", "KMX", "RVTY", "CRL", "MGM", "ROL", "GEN", "JKHY", "WRK", "LNT", "KIM", "TAP", "AES", "EVRG", "IPG", "EMN", "SJM", "PODD", "JNPR", "ALLE", "FFIV", "HII", "UDR", "LW", "QRVO", "NI", "CPT", "TECH", "APA", "AOS", "BBWI", "MOS", "UHS", "CTLT", "INCY", "TFX", "WYNN", "HRL", "TPR", "PAYC", "NWSA", "REG", "DAY", "AIZ", "HSIC", "SOLV", "MTCH", "GL", "BF.B", "CZR", "AAL", "BXP", "CPB", "MKTX", "CHRW", "PNW", "GNRC", "BWA", "NCLH", "RHI", "ETSY", "FOXA", "BEN", "IVZ", "FMC", "FRT", "HAS", "DVA", "CMA", "BIO", "RL", "MHK",]



resultados_modelos = {}

for ticker in lista_tickers:
    print(f"Procesando {ticker}...")
    try:
        # Asumiendo que la función regresion devuelve el p-valor de la constante
        p_valor = regresion(ticker)
        resultados_modelos[ticker] = p_valor  # Almacenando el p-valor en el diccionario con la clave del ticker
    except Exception as e:
        print(f"Error al procesar {ticker}: {e}")
        resultados_modelos[ticker] = None  # Almacenando None si ocurre un error durante la regresión

# Ahora tienes todos tus p-valores almacenados en resultados_modelos

Procesando MSFT...


[*********************100%%**********************]  1 of 1 completed


Procesando AAPL...


[*********************100%%**********************]  1 of 1 completed


Procesando NVDA...


...

[*********************100%%**********************]  1 of 1 completed


Procesando RL...


[*********************100%%**********************]  1 of 1 completed


Procesando MHK...


[*********************100%%**********************]  1 of 1 completed


In [24]:
print(resultados_modelos)

{'MSFT': 0.7610769048900914, 'AAPL': 0.03856704140605782, 'NVDA': 0.12763743174124173, 'AMZN': 0.3798545960130929, 'META': 0.20859634351145373, 'GOOGL': 0.03700508759251584, 'GOOG': 0.036446063121147725, 'BRK.B': None, 'LLY': 0.348207335373667, 'AVGO': 0.009157782295533318, 'JPM': 0.07764420904623251, 'TSLA': 0.08157307048002571, 'XOM': 0.23684292829095135, 'V': 0.10902675923810166, 'UNH': 0.07540713735942739, 'MA': 0.013055200863906918, 'PG': 0.9488804740318064, 'JNJ': 0.970200805855493, 'HD': 0.5763290489100139, 'MRK': 0.5284415310984175, 'COST': 0.30613410688384374, 'ABBV': 0.41112522018431497, 'CRM': 0.17821126629573675, 'CVX': 0.3193105968871366, 'AMD': 0.6558564578290055, 'NFLX': 0.09129185126669698, 'BAC': 0.008913540879175112, 'WMT': 0.7439836975599159, 'PEP': 0.6468545182950454, 'KO': 0.4206994299995721, 'LIN': 0.4555535819619301, 'TMO': 0.12678049049397996, 'ADBE': 0.3372738345371049, 'DIS': 0.309788361528619, 'ACN': 0.23548288522107266, 'WFC': 0.04951491481689551, 'ORCL': 0.

ValueError: If using all scalar values, you must pass an index

In [ ]:
from datetime import datetimedf_combinado.to_csv('ex.csv', sep = ";", index=True)